In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

SILVER_TRANSFORM_CONFIG_PATH = (
    BASE_PATH + "metadata/silver_column_transform_config/"
)
dbutils.widgets.text(
    "dataset_name",
    "google_ads_campaigns",
    "DATASET NAME"
)

DATASET_NAME = dbutils.widgets.get("dataset_name").strip()

if not DATASET_NAME:
    raise ValueError("dataset_name cannot be empty.")

print("Schema config:", SCHEMA_CONFIG_PATH)
print("Transformation config:", SILVER_TRANSFORM_CONFIG_PATH)

Schema config: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/schema_config/
Transformation config: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/silver_column_transform_config/


In [0]:
if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):
    raise RuntimeError(
        f"Schema config not found: {SCHEMA_CONFIG_PATH}"
    )

schema_config_df = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

active_schema = (
    schema_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == DATASET_NAME)
        & (F.col("is_active") == True)
    )
    .select("schema_ddl", "schema_version", "source_system")
    .collect()
)

if len(active_schema) != 1:
    raise ValueError(
        f"Expected exactly one active schema for {DATASET_NAME}, "
        f"found {len(active_schema)}"
    )

SCHEMA_DDL = active_schema[0]["schema_ddl"]
SCHEMA_VERSION = active_schema[0]["schema_version"]
SOURCE_SYSTEM = active_schema[0]["source_system"]

SOURCE_SCHEMA = StructType.fromDDL(SCHEMA_DDL)

SOURCE_FIELDS = {
    field.name: field.dataType
    for field in SOURCE_SCHEMA.fields
}

STRING_COLUMNS = []

for field in SOURCE_SCHEMA.fields:
    if field.dataType == StringType():
        STRING_COLUMNS.append(field.name)

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Schema version:", SCHEMA_VERSION)
print("Registered columns:", len(SOURCE_FIELDS))
print("String columns:", STRING_COLUMNS)
print(SOURCE_FIELDS)

Dataset: google_ads_campaigns
Source system: google_ads
Schema version: 1
Registered columns: 18
String columns: ['customer_id', 'campaign_id', 'campaign_resource_name', 'campaign_name', 'campaign_status', 'advertising_channel_type', 'advertising_channel_sub_type', 'campaign_budget_resource_name', 'bidding_strategy_type', 'start_date', 'end_date', 'campaign_primary_status', 'serving_status', 'target_country', 'business_brand', 'product_category', 'business_objective', 'source_extract_timestamp']
{'customer_id': StringType(), 'campaign_id': StringType(), 'campaign_resource_name': StringType(), 'campaign_name': StringType(), 'campaign_status': StringType(), 'advertising_channel_type': StringType(), 'advertising_channel_sub_type': StringType(), 'campaign_budget_resource_name': StringType(), 'bidding_strategy_type': StringType(), 'start_date': StringType(), 'end_date': StringType(), 'campaign_primary_status': StringType(), 'serving_status': StringType(), 'target_country': StringType(), 'bu

In [0]:
transform_config_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("source_column", StringType(), False),
    StructField("target_column", StringType(), False),

    StructField("transformation_type", StringType(), False),
    StructField("transformation_parameters", StringType(), True),

    StructField("execution_order", IntegerType(), False),
    StructField("is_active", BooleanType(), False),

    StructField("description", StringType(), True),

    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

print("Transformation configuration schema defined.")

Transformation configuration schema defined.


In [0]:
if not DeltaTable.isDeltaTable(
    spark,
    SILVER_TRANSFORM_CONFIG_PATH
):

    empty_df = spark.createDataFrame(
        [],
        transform_config_schema
    )

    (
        empty_df.write
        .format("delta")
        .mode("ignore")
        .save(SILVER_TRANSFORM_CONFIG_PATH)
    )

    print("Transformation configuration table created.")

else:
    print("Transformation configuration table already exists.")

Transformation configuration table already exists.


In [0]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)

transformations = []

# Explicitly configured transformation requirements
transformation_rules = [
    ("campaign_name", "TRIM", 1),
    ("campaign_name", "NULL_IF_EMPTY", 2),

    ("business_brand", "TRIM", 1),
    ("business_brand", "NULL_IF_EMPTY", 2),

    ("product_category", "TRIM", 1),
    ("product_category", "NULL_IF_EMPTY", 2),

    ("business_objective", "TRIM", 1),
    ("business_objective", "NULL_IF_EMPTY", 2)
]

for column_name, transformation_type, execution_order in transformation_rules:

    # Validate that the source column exists in schema_config.
    if column_name not in SOURCE_FIELDS:
        raise ValueError(
            f"Column '{column_name}' is not registered in schema_config."
        )

    transformations.append((
        DATASET_NAME,
        column_name,
        column_name,
        transformation_type,
        None,
        execution_order,
        True,
        f"{transformation_type} applied to {column_name}",
        now,
        now
    ))

if not isinstance(SOURCE_FIELDS[column_name], StringType):
    raise TypeError(
        f"{transformation_type} requires a string column: {column_name}"
    )
if not transformations:
    raise ValueError("No transformation rules configured.")

print("Total transformation rules:", len(transformations))

source_df = spark.createDataFrame(
    transformations,
    transform_config_schema
)

display(source_df.orderBy("source_column", "execution_order"))

Total transformation rules: 8


dataset_name,source_column,target_column,transformation_type,transformation_parameters,execution_order,is_active,description,created_at,updated_at
google_ads_campaigns,business_brand,business_brand,TRIM,null,1,true,TRIM applied to business_brand,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_brand,business_brand,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_brand,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,TRIM,null,1,true,TRIM applied to business_objective,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_objective,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,TRIM,null,1,true,TRIM applied to campaign_name,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to campaign_name,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,TRIM,null,1,true,TRIM applied to product_category,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to product_category,2026-10-31T13:29:05.905Z,2026-10-31T13:29:05.905Z


In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forPath(
    spark,
    SILVER_TRANSFORM_CONFIG_PATH
)

(
    target_table.alias("target")
    .merge(
        source_df.alias("source"),
        """
        target.dataset_name = source.dataset_name
        AND target.source_column = source.source_column
        AND target.transformation_type = source.transformation_type
        """
    )
    .whenMatchedUpdate(set={
        "target_column": "source.target_column",
        "transformation_parameters": "source.transformation_parameters",
        "execution_order": "source.execution_order",
        "is_active": "source.is_active",
        "description": "source.description",
        "updated_at": "source.updated_at"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("Transformation configurations registered successfully!")

Transformation configurations registered successfully!


In [0]:
registered_df = (
    spark.read.format("delta")
    .load(SILVER_TRANSFORM_CONFIG_PATH)
    .filter(
        (F.col("dataset_name") == DATASET_NAME) &
        (F.col("is_active") == True)
    )
)

print("Active transformation rules:", registered_df.count())

display(
    registered_df.orderBy("source_column", "execution_order")
)

Active transformation rules: 8


dataset_name,source_column,target_column,transformation_type,transformation_parameters,execution_order,is_active,description,created_at,updated_at
google_ads_campaigns,business_brand,business_brand,TRIM,null,1,true,TRIM applied to business_brand,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_brand,business_brand,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_brand,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,TRIM,null,1,true,TRIM applied to business_objective,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,business_objective,business_objective,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to business_objective,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,TRIM,null,1,true,TRIM applied to campaign_name,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,campaign_name,campaign_name,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to campaign_name,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,TRIM,null,1,true,TRIM applied to product_category,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
google_ads_campaigns,product_category,product_category,NULL_IF_EMPTY,null,2,true,NULL_IF_EMPTY applied to product_category,2026-10-31T13:18:26.059Z,2026-10-31T13:29:05.905Z
